# 06 The actual screen

Everything built so far comes together here.

**What happens, in order:**

```
get the jamu compounds out of COCONUT        (six Indonesian plants)
        |
check each one against Model A's gate        -> outside? report Unknown, stop
        |
run Model A on the rest                      -> condition 1
        |
pair each compound with every drug           (the panel from notebook 05)
        |
apply the three-condition rules              -> High / Medium / Unknown / not flagged
```

**The three conditions again, briefly:**

1. the jamu compound jams CYP3A4 — *Model A predicts this*
2. the drug is cleared by CYP3A4 — *looked up in lab data*
3. the drug causes heart trouble when it builds up — *lab data, or Model B*

All three have to hold. Break one link and there is no risk to report.

**What comes out** is a ranked list of things worth testing in a lab. It is not a list of dangerous combinations, and the score attached to each one orders pairs within a tier and nothing else.

In [1]:
import sys
import importlib
import os

sys.path.append(os.path.abspath(os.path.join("..", "src")))

import numpy as np
import pandas as pd
import joblib

import features
import applicability
import combine
import coconut
import explain

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")

# Python only reads a file from src/ the FIRST time it is imported. After that
# it serves a cached copy, so any edit you make to src/ is invisible to a
# notebook that has already run this cell.
#
# The symptom is confusing: an AttributeError saying a function does not exist,
# when you can see it sitting right there in the file. Reloading here means
# "Run All" always uses what is actually on disk.
for module in (features, applicability, coconut, explain, combine):
    importlib.reload(module)

## Get the jamu compounds out of COCONUT

Two problems here, and the second is the one that matters.

**The easy problem: the file is huge.** A ~240 MB zip holding a ~700 MB spreadsheet. Loading it all at once will run a laptop out of memory, so `src/coconut.py` reads it in chunks and throws each chunk away after filtering it.

**The real problem: plant names are a mess.** COCONUT's organism field is free text contributed by many different databases, so one plant turns up under its current official name, its old names, and several spellings. Search for only the official name and most rows vanish.

That failure is dangerous because it looks like a result: "this plant has no known compounds" rather than "my search was too narrow". So `JAMU_SPECIES` lists every variant for each plant. *Curcuma domestica* and *Curcuma zanthorrhiza* are the two that would otherwise lose the most rows.

In [2]:
for canonical, variants in coconut.JAMU_SPECIES.items():
    print(f"{canonical}")
    for v in variants:
        print(f"    matches: {v}")

Curcuma longa
    matches: curcuma longa
    matches: curcuma domestica
    matches: curcuma domestica valeton
Curcuma xanthorrhiza
    matches: curcuma xanthorrhiza
    matches: curcuma zanthorrhiza
    matches: curcuma xanthorriza
Zingiber officinale
    matches: zingiber officinale
Andrographis paniculata
    matches: andrographis paniculata
    matches: justicia paniculata
Phyllanthus niruri
    matches: phyllanthus niruri
Orthosiphon aristatus
    matches: orthosiphon aristatus
    matches: orthosiphon stamineus
    matches: orthosiphon spicatus


In [3]:
# find_bulk finds whichever COCONUT release was downloaded, zipped or not, and
# stops with the download link if there is none. It searches by pattern rather
# than a fixed filename, because COCONUT reissues the file every month with the
# date in its name.
COCONUT_BULK = coconut.find_bulk()
HERB_CSV = os.path.join(PROCESSED_DIR, "herb_compounds.csv")

# Saved after the first run, because scanning 700 MB takes a few minutes and
# the answer does not change. If you edit JAMU_SPECIES, delete
# herb_compounds.csv or you will keep getting the old list back.
if os.path.exists(HERB_CSV):
    herb_df = pd.read_csv(HERB_CSV)
    print(f"loaded {len(herb_df)} cached compounds")
else:
    print("scanning", os.path.basename(COCONUT_BULK))
    herb_df, stats = coconut.filter_bulk(COCONUT_BULK)
    herb_df.to_csv(HERB_CSV, index=False)
    print(f"scanned {stats['rows_scanned']:,} rows -> {stats['unique_compounds']:,} compounds")

print()
print(coconut.species_counts(herb_df).to_string())

loaded 1861 cached compounds

Zingiber officinale        911
Curcuma longa              518
Andrographis paniculata    361
Orthosiphon aristatus      258
Curcuma xanthorrhiza       104
Phyllanthus niruri          51


## Turn the jamu compounds into numbers

Same featurising as notebook 01, so the model sees these molecules in exactly the same format it was trained on.

In [4]:
herb_fps, kept = features.smiles_list_to_matrix(herb_df["canonical_smiles"].tolist())
dropped = len(herb_df) - len(kept)

print("herb fingerprints:", herb_fps.shape)
print(f"unparseable SMILES dropped: {dropped}")
if dropped:
    print("dropped identifiers:",
          herb_df.iloc[[i for i in range(len(herb_df)) if i not in set(kept)]]
          ["identifier"].tolist()[:20])

# Drop the same rows from the names table that got dropped from the numbers.
#
# This has to happen or everything after it is wrong in the worst way: row 5 of
# the numbers would get paired with row 5 of the names, when it is really
# molecule 7. Every result would be attributed to the wrong compound, and
# nothing would look broken.
herb_df = herb_df.iloc[kept].reset_index(drop=True)
assert len(herb_df) == herb_fps.shape[0]

herb fingerprints: (1861, 2058)
unparseable SMILES dropped: 0


## The gate: which of these can the model honestly judge?

The gate was built in notebook 02, from Model A's **training** molecules only.

**Expect a lot of these compounds to fall outside it, and do not treat that as a problem.** Model A learned from synthetic, drug-like molecules. These are plant compounds, often built quite differently. Where the model has no comparable chemistry to go on, the honest answer is Unknown.

A screen that returned a confident number for all 1,861 would look more impressive and be worth less. This number is one of the project's actual findings.

In [5]:
domain = applicability.load_domain(os.path.join(PROCESSED_DIR, "domain_a.npz"))
# The gate needs yes/no fragment boxes. herb_fps is the model's format instead:
# fragment COUNTS plus ten whole-molecule numbers.
#
# Here the shapes luckily disagree (2048 vs 2058) so it would crash. But even
# if they matched it would be wrong, because Tanimoto similarity only means
# anything for yes/no sets. Counts are not yes/no, and molecular weight in the
# hundreds would drown out every fragment column.
herb_bits = features.bits_from_matrix(herb_fps)
verdicts, scores = applicability.check_molecules(herb_bits, domain)

herb_df["domain_status"] = verdicts
herb_df["domain_score"] = scores

n_outside = sum(1 for v in verdicts if v == applicability.OUTSIDE)
print(f"domain floor (from Model A training data): {domain['floor']:.4f}")
print(f"herb compounds inside:  {len(verdicts) - n_outside}")
print(f"herb compounds outside: {n_outside}  ({n_outside / len(verdicts):.1%})")
print()
print("Outside the domain -> Unknown tier. Model A is not run on those at all,")
print("because a number the tier logic then discards tends to get quoted anyway.")

domain floor (from Model A training data): 0.2827
herb compounds inside:  1009
herb compounds outside: 852  (45.8%)

Outside the domain -> Unknown tier. Model A is not run on those at all,
because a number the tier logic then discards tends to get quoted anyway.


## Condition 1: run Model A, but only where it is allowed to answer

Compounds outside the gate are skipped entirely rather than scored and then ignored. A number that gets discarded two steps later has a habit of being quoted anyway.

In [6]:
model_a = joblib.load(os.path.join(PROCESSED_DIR, "model_a.joblib"))
model_b = joblib.load(os.path.join(PROCESSED_DIR, "model_b.joblib"))

inside_mask = np.array([v == applicability.INSIDE for v in verdicts])
if inside_mask.any():
    inside_probs = model_a.predict_proba(herb_fps[inside_mask])[:, 1]
    print(f"Model A run on {inside_mask.sum()} in-domain compounds")
    print(f"predicted inhibitors at p >= {combine.HIGH_CONF}: "
          f"{(inside_probs >= combine.HIGH_CONF).sum()}")
    print(f"borderline ({combine.LOW_CONF} - {combine.HIGH_CONF}): "
          f"{((inside_probs > combine.LOW_CONF) & (inside_probs < combine.HIGH_CONF)).sum()}")
    print(f"predicted non-inhibitors at p <= {combine.LOW_CONF}: "
          f"{(inside_probs <= combine.LOW_CONF).sum()}")
else:
    print("no compounds inside the domain")

Model A run on 1009 in-domain compounds
predicted inhibitors at p >= 0.7: 117
borderline (0.3 - 0.7): 201
predicted non-inhibitors at p <= 0.3: 691


## The jamu side on its own

Condition 1 is finished here: every jamu compound now has a verdict, either a probability or Unknown. **That is already a result**, and it does not depend on the drug side at all.

Ranking covers only the compounds inside the gate. An Unknown compound has no probability worth ranking, so leaving it in the list would sort it to the bottom and make it look like a low-risk compound, which is exactly the wrong message.

In [7]:
herb_df["p_inhibits"] = np.nan
herb_df.loc[inside_mask, "p_inhibits"] = inside_probs

herb_ranked = (herb_df[inside_mask]
               .sort_values("p_inhibits", ascending=False)
               .reset_index(drop=True))
herb_ranked.to_csv(os.path.join(PROCESSED_DIR, "jamu_scores.csv"), index=False)

columns = [c for c in ["identifier", "name", "matched_species",
                       "p_inhibits", "domain_score"] if c in herb_ranked.columns]
print(f"top 15 of {len(herb_ranked)} in-domain compounds:\n")
print(herb_ranked[columns].head(15).to_string(index=False, max_colwidth=32))

top 15 of 1009 in-domain compounds:

  identifier                             name       matched_species  p_inhibits  domain_score
CNP0212422.3                        Hinokinin    Phyllanthus niruri    0.979366      0.404436
CNP0128342.2                      rac-Cubebin   Zingiber officinale    0.976182      0.354985
CNP0099030.1                     CNP0099030.1   Zingiber officinale    0.958928      0.412022
CNP0609779.0                     CNP0609779.0   Zingiber officinale    0.953082      0.346128
CNP0297841.1                     CNP0297841.1   Zingiber officinale    0.948720      0.411581
CNP0297841.2                     CNP0297841.2   Zingiber officinale    0.948720      0.411581
CNP0611431.0                     CNP0611431.0   Zingiber officinale    0.947345      0.321664
CNP0402510.1                     CNP0402510.1   Zingiber officinale    0.945506      0.449526
CNP0402510.2 (3S,5S)-Diacetoxy-1,7-bis-(4-...   Zingiber officinale    0.945506      0.449526
CNP0315129.1           

### Why, not just how much

A score on its own is not useful: nobody can act on "0.98". So each top compound also gets the chemical reason behind its score, using the same explainer built in notebook 04.

- **phi > 0** → this piece pushed towards jamming the enzyme (a risk driver)
- **phi < 0** → this piece pulled away from it (a protective group)

Now a flagged compound comes with something a chemist can check against what is already known.

In [8]:
for _, row in herb_ranked.head(3).iterrows():
    result = explain.explain_molecule(model_a, row["canonical_smiles"], top_n=3)
    if result is None:
        continue
    label = str(row.get("name") or row["identifier"])
    print(f"{label}  ({row['identifier']}, {row['matched_species']})")
    print(f"  P(inhibits CYP3A4) = {result['probability']:.3f}")
    for fragment in result["fragments"]:
        print(f"    phi {fragment['shap']:+.4f}  {fragment['role']:13s} "
              f"{fragment['description']}")
    print()

Hinokinin  (CNP0212422.3, Phyllanthus niruri)
  P(inhibits CYP3A4) = 0.979
    phi +1.5285  risk driver   14 atoms (C/O) - 8 aromatic atom(s); ring size 5, 6; oxygen present
    phi +1.1756  risk driver   10 atoms (C/O) - 4 aromatic atom(s); ring size 5, 6; oxygen present
    phi +0.3309  risk driver   7 atoms (C) - 6 aromatic atom(s); ring size 5, 6

rac-Cubebin  (CNP0128342.2, Zingiber officinale)
  P(inhibits CYP3A4) = 0.976
    phi +1.5166  risk driver   14 atoms (C/O) - 8 aromatic atom(s); ring size 5, 6; oxygen present
    phi +0.9319  risk driver   10 atoms (C/O) - 4 aromatic atom(s); ring size 5, 6; oxygen present
    phi +0.3595  risk driver   12 atoms (C/O) - 8 aromatic atom(s); ring size 5, 6; oxygen present



CNP0099030.1  (CNP0099030.1, Zingiber officinale)
  P(inhibits CYP3A4) = 0.959
    phi +0.5190  risk driver   6 atoms (C/O) - 2 aromatic atom(s); ring size 6; oxygen present
    phi +0.3523  risk driver   10 atoms (C/O) - oxygen present
    phi +0.3489  risk driver   2 atoms (C)



---
## Now bring in the drugs

Conditions 2 and 3 describe the **drug**, not the plant compound, and both come from the panel built in notebook 05.

If that panel is missing, the cell below stops and says so. It does not carry on with an empty drug list, because that would produce an empty screen that reads as "no risky pairs found" — a reassuring answer that is really just a missing file.

In [9]:
SCREEN_DRUGS = os.path.join(PROCESSED_DIR, "screen_drugs.csv")
HAVE_DRUGS = os.path.exists(SCREEN_DRUGS)

if not HAVE_DRUGS:
    # Not an error. Everything above is finished and saved; only the drug side
    # is missing, and that comes from notebook 05.
    print("screen_drugs.csv missing, so the pairing step is skipped.")
    print("Run notebook 05 to build the drug panel, then re-run this cell.")
else:
    drug_df = pd.read_csv(SCREEN_DRUGS)
    # Both conditions come out of the one panel file, so they cannot disagree
    # about which drugs are in the screen.
    substrate_lookup, qt_lookup = combine.lookups_from_panel(drug_df)

    # Structures come with the panel, so there is no extra file to fill in.
    #
    # Anything RDKit cannot read becomes None, and combine.py then reports that
    # drug's condition 3 as "unavailable". Padding it with zeros instead would
    # be a molecule with no atoms, and Model B would still hand back a
    # confident-looking number for it.
    drug_fps = [features.smiles_to_fp(s) for s in drug_df["smiles"].astype(str)]

    results = combine.screen_pairs(
        herb_df, drug_df, model_a, model_b,
        substrate_lookup, qt_lookup, domain,
        herb_fps=herb_fps, herb_bits=herb_bits, drug_fps=drug_fps,
    )
    print(f"flagged pairs: {len(results)}\n")
    print(results["tier"].value_counts().to_string())

flagged pairs: 86814

tier
Medium    73242
High      13572


In [10]:
if not HAVE_DRUGS:
    print("no screening results yet, see the cell above")
else:
    # The full pair table is deliberately NOT saved, and it was 83 MB when it
    # was. The reason: conditions 2 and 3 depend only on the drug and condition
    # 1 only on the compound, so these rows are just every compound crossed
    # with every drug: ~116 nearly identical rows per compound, differing only
    # in the drug name.
    #
    # The two things it is built from are both saved (flagged_compounds.csv and
    # screen_drugs.csv), and rebuilding the pairs from them is a one line merge.
    print(f"flagged pairs this session: {len(results)}")
    print()
    print("best pair per compound, top 15 compounds:")
    cols = ["herb_name", "drug", "tier", "cond1_prob", "cond3_source", "rank_score"]
    print(results.drop_duplicates("herb_identifier")[cols].head(15)
          .to_string(index=False, max_colwidth=30))

flagged pairs this session: 86814

best pair per compound, top 15 compounds:
                     herb_name       drug tier  cond1_prob cond3_source  rank_score
                     Hinokinin amiodarone High    0.979366      qt list    0.979366
                   rac-Cubebin amiodarone High    0.976182      qt list    0.976182
                  CNP0099030.1 amiodarone High    0.958928      qt list    0.958928
                  CNP0609779.0 amiodarone High    0.953082      qt list    0.953082
                  CNP0297841.1 amiodarone High    0.948720      qt list    0.948720
                  CNP0297841.2 amiodarone High    0.948720      qt list    0.948720
                  CNP0611431.0 amiodarone High    0.947345      qt list    0.947345
                  CNP0402510.1 amiodarone High    0.945506      qt list    0.945506
(3S,5S)-Diacetoxy-1,7-bis-(... amiodarone High    0.945506      qt list    0.945506
                  CNP0315129.1 amiodarone High    0.943913      qt list    0.943913

## Report by compound, not by pair

Here is a trap in reading these results.

Condition 1 belongs to the compound; conditions 2 and 3 belong to the drug. So **every pair sharing the same jamu compound gets the same score** and they all tie. Sort the raw pair list and you get one compound repeated 116 times, which tells you nothing after the first row. That is what the top-20 table looked like before this cell existed.

**The compound is the unit that means something.** Each one is a single laboratory experiment someone could actually run. A compound's pair count only reflects how many drugs happen to be in the panel, so it is not a measure of how risky it is.

In [11]:
if HAVE_DRUGS:
    rollup = (results.groupby(["herb_identifier", "herb_name"])
              .agg(high_pairs=("tier", lambda t: int((t == combine.HIGH).sum())),
                   medium_pairs=("tier", lambda t: int((t == combine.MEDIUM).sum())),
                   p_inhibits=("cond1_prob", "max"))
              .reset_index()
              .sort_values(["high_pairs", "p_inhibits"], ascending=False))

    plants = herb_df.set_index("identifier")["matched_species"]
    rollup["plant"] = rollup["herb_identifier"].map(plants)
    rollup.to_csv(os.path.join(PROCESSED_DIR, "flagged_compounds.csv"), index=False)

    high = rollup[rollup["high_pairs"] > 0]
    print(f"distinct jamu compounds reaching High tier : {len(high)}")
    print(f"distinct compounds flagged at all          : {len(rollup)}")
    print(f"distinct drugs involved in a High pair     : "
          f"{results.loc[results['tier'] == combine.HIGH, 'drug'].nunique()}")
    print()
    print("High-tier compounds per plant:")
    print(high["plant"].str.split("|").explode().value_counts().to_string())
    print()
    print("top 15 compounds by predicted CYP3A4 inhibition:")
    print(rollup.head(15)[["herb_name", "plant", "p_inhibits", "high_pairs"]]
          .to_string(index=False, max_colwidth=30))
    print()
    print("saved flagged_compounds.csv")

distinct jamu compounds reaching High tier : 117
distinct compounds flagged at all          : 318
distinct drugs involved in a High pair     : 116

High-tier compounds per plant:
plant
Zingiber officinale        84
Orthosiphon aristatus      13
Andrographis paniculata    13
Phyllanthus niruri          8
Curcuma longa               7
Curcuma xanthorrhiza        4

top 15 compounds by predicted CYP3A4 inhibition:
                     herb_name                 plant  p_inhibits  high_pairs
                     Hinokinin    Phyllanthus niruri    0.979366         116
                   rac-Cubebin   Zingiber officinale    0.976182         116
                  CNP0099030.1   Zingiber officinale    0.958928         116
                  CNP0609779.0   Zingiber officinale    0.953082         116
                  CNP0297841.1   Zingiber officinale    0.948720         116
                  CNP0297841.2   Zingiber officinale    0.948720         116
                  CNP0611431.0   Zingiber offi

## External check against HerbCYP

Source: [github.com/Gumgo91/HerbCYP](https://github.com/Gumgo91/HerbCYP). They publish their
own CYP3A4 predictions, so comparing on the compounds that appear in both sets is a free
external check.

Agreement is weak independent support. **Disagreement is reported as it stands**, not quietly
dropped, because neither set of predictions is ground truth.

In [12]:
# TODO once their prediction file is downloaded into data/raw/:
#   1. load it, match on InChIKey rather than on name or SMILES string
#      (the same molecule has many valid SMILES, so a string join would
#       under-report agreement)
#   2. report the confusion matrix between our call and theirs
#   3. report the compounds where the two disagree, with both probabilities

## Reading the output

Every row is a hypothesis for laboratory testing. A High tier row does not say a jamu product is dangerous: jamu is a multi-compound extract, and a compound appearing in a plant says nothing about how much of it is in a product or how much reaches the liver. The limitations section of the README applies to this table in full.